In [ ]:
import pyamg
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

In [ ]:
n = 99
A = pyamg.gallery.poisson((n,), format='csr')
b = np.random.rand(n)
x0 = np.random.rand(n)

In [ ]:
# 0, 1, .... , n, n+1
# G(w) = |1 - 2 w sin^2(k pi / (2(n+1))|  1 <= k <= n
def G(omega):
    n = 100
    K = np.arange(1,n+1)
    eigs = np.abs(1-2*omega*np.sin(K*np.pi/(2*(n+1)))**2)
    return eigs

omegalist = ['1', '2/3', '4/3', '4/5']
fig, ax = plt.subplots()
for omega in omegalist:
    eigs = G(Fraction(omega))
    ax.plot(eigs, label=f'{omega}')
ax.vlines(33,0,1,color='k', linestyle='--', lw=0.5)
ax.hlines(.6,0,100,'b', linestyle=':', lw=0.5)
ax.legend()

fig, ax = plt.subplots()
w = np.mgrid[0:2:100j]
ax.plot(w, np.abs(1-w))
ax.plot(w, np.abs(1-2*w))

In [ ]:
smoother = ('jacobi', {'omega': 2/3, 'withrho': False})
ml = pyamg.ruge_stuben_solver(A, max_levels=2,
                              presmoother=smoother,
                              postsmoother=smoother, keep=True,
                             )

res = []
x  = ml.solve(b, x0, tol=1e-10, residuals=res)
res = np.array(res)
print(res[1:]/res[:-1])
print(ml)

In [ ]:
print(ml.levels[0].splitting)

In [ ]:
#smoother = ('jacobi', {'omega': 5/5, 'withrho': False})
smoother = ('gauss_seidel')
ml = pyamg.smoothed_aggregation_solver(A, max_levels=2,
                              #presmoother=smoother,
                              #postsmoother=smoother,
                              strength=('symmetric', {'theta': 0.1}), keep=True,
                             )

res = []
x  = ml.solve(b, x0, tol=1e-10, residuals=res)
res = np.array(res)
print(res[1:]/res[:-1])
print(ml)
print(ml.levels[0].AggOp.toarray())

In [ ]:
nx = 30
A = pyamg.gallery.poisson((nx,nx), format='csr', type='FE')
n = A.shape[0]
b = np.random.rand(n)
x0 = np.random.rand(n)

In [ ]:
smoother = ('jacobi', {'omega': 4/5, 'withrho': False})
ml = pyamg.ruge_stuben_solver(A, max_levels=2,
                              presmoother=smoother,
                              postsmoother=smoother,
                             )

res = []
x  = ml.solve(b, x0, tol=1e-10, residuals=res)
res = np.array(res)
print(res[1:]/res[:-1])
print(ml)

In [ ]:
omegalist = np.linspace(0.1, 1.5, 100)
factors = []
for omega in omegalist:
    smoother = ('jacobi', {'omega': omega, 'withrho': False})
    ml = pyamg.smoothed_aggregation_solver(A, max_levels=2,
                                  presmoother=smoother,
                                  postsmoother=smoother,
                                  strength=('symmetric', {'theta': 0}),
                                  #improve_candidates=None
                                 )

    res = []
    x  = ml.solve(b, x0, tol=1e-10, residuals=res, maxiter=20)
    res = np.array(res)
    factor = res[1:]/res[:-1]
    factors.append(factor[0])

In [ ]:
plt.plot(omegalist, factors)

In [ ]:
factors